# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Fundamentals of Estimation

This notebook accompanies Chapter 5, Sections 5.1--5.2. We distinguish an
estimand, an estimator, and an observed estimate, then study sampling
distributions, bias, standard error, variance, and mean squared error.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(606)


## What are we trying to estimate?

Let $X_1,\ldots,X_n$ be independent
$\mathrm{Bernoulli}(\theta)$ random variables. The **estimand** is the
fixed but unknown success probability $\theta$. The rule

$$
\widehat\Theta_n(X_1,\ldots,X_n)=\overline X_n
$$

is an **estimator**, hence a random variable before observing data. Applying
the rule to observed values $x_1,\ldots,x_n$ produces a numerical
**estimate**. An estimator may use the data and known model features but may
not use the unknown value of $\theta$.


In [ ]:
observed_data = np.array([1, 0, 1, 1, 0])
observed_estimate = observed_data.mean()
print("observed estimate =", observed_estimate)


## How an estimator changes from sample to sample

The sampling distribution describes how an estimator varies over repeated
samples from one fixed parameter value. For an estimator $T$ of $\theta$,

$$
\begin{aligned}
\operatorname{bias}_\theta(T)&=\mathbb E_\theta[T]-\theta,\\
\operatorname{se}_\theta(T)&=\sqrt{\operatorname{Var}_\theta(T)},\\
\operatorname{MSE}_\theta(T)&=\mathbb E_\theta[(T-\theta)^2]\\
&=\operatorname{Var}_\theta(T)+\operatorname{bias}_\theta(T)^2.
\end{aligned}
$$

For the Bernoulli sample mean, the bias is 0, the variance and MSE are
$\theta(1-\theta)/n$, and the standard error is the square root of this
quantity.


## Can a biased estimator be better?

Let $K=\sum_i X_i$ and consider the smoothed estimator

$$
\widetilde\Theta_n=\frac{K+1}{n+2}.
$$

It has

$$
\operatorname{bias}_\theta(\widetilde\Theta_n)
=\frac{1-2\theta}{n+2},\qquad
\operatorname{Var}_\theta(\widetilde\Theta_n)
=\frac{n\theta(1-\theta)}{(n+2)^2}.
$$

Its variance is smaller than that of the sample mean, but it generally has
nonzero bias. MSE puts both effects on the same scale.


In [ ]:
theta = 0.30
repetitions = 20_000
sample_sizes = (10, 50, 200)


def theoretical_summaries(n, theta):
    mean_bias = 0.0
    mean_variance = theta * (1 - theta) / n
    smooth_bias = (1 - 2 * theta) / (n + 2)
    smooth_variance = n * theta * (1 - theta) / (n + 2) ** 2
    return {
        "mean": (mean_bias, np.sqrt(mean_variance), mean_variance),
        "smooth": (
            smooth_bias,
            np.sqrt(smooth_variance),
            smooth_variance + smooth_bias**2,
        ),
    }


for n in sample_sizes:
    counts = rng.binomial(n, theta, size=repetitions)
    estimators = {
        "mean": counts / n,
        "smooth": (counts + 1) / (n + 2),
    }
    theory = theoretical_summaries(n, theta)
    print(f"n={n}")
    for name, estimates in estimators.items():
        simulated = (
            estimates.mean() - theta,
            estimates.std(ddof=0),
            np.mean((estimates - theta) ** 2),
        )
        print(f"  {name:6s} simulated bias/SE/MSE: {np.round(simulated, 5)}")
        print(f"         theory bias/SE/MSE:    {np.round(theory[name], 5)}")


In [ ]:
n = 20
counts = rng.binomial(n, theta, size=repetitions)
sample_mean_estimates = counts / n
smoothed_estimates = (counts + 1) / (n + 2)

fig, ax = plt.subplots(figsize=(7, 3.8))
bins = np.linspace(0, 0.75, 31)
ax.hist(sample_mean_estimates, bins=bins, density=True, alpha=0.5, label="sample mean")
ax.hist(smoothed_estimates, bins=bins, density=True, alpha=0.5, label="smoothed")
ax.axvline(theta, color="black", linestyle="--", label=r"estimand $\theta$")
ax.set(xlabel="estimate", ylabel="density", title="Monte Carlo sampling distributions")
ax.legend()
plt.show()


Monte Carlo summaries approximate properties of a sampling distribution; they
are not definitions of bias or standard error. The independent simulations
above all use the same fixed value $\theta=0.3$.

Both estimators are consistent: for the sample mean, bias is zero and
standard error tends to zero; for the smoothed estimator, both its bias and
standard error tend to zero. The MSE bound

$$
\mathbb P_\theta(|T_n-\theta|>\varepsilon)
\leq \operatorname{MSE}_\theta(T_n)/\varepsilon^2
$$

then gives convergence in probability whenever the MSE tends to zero.


## Try it yourself

1. Derive the displayed bias and variance formulas for
   $(K+1)/(n+2)$ from $K\sim\mathrm{Binomial}(n,\theta)$.
2. For $n=10$, compare the two theoretical MSEs on a grid of
   $\theta\in[0,1]$. Plot where smoothing helps and where it hurts.
3. Explain why an observed estimate such as $0.6$ has no sampling variance,
   while the estimator that produced it has a sampling variance.
